# 14 — Benamou–Brenier and Otto

Close Module 3 by viewing the $W_2$ geodesic two ways — McCann's explicit affine map, and Benamou–Brenier's least-energy flow — and add the last rows Module 3 brings to the dictionary that carries transport into the quantum world.

**Prerequisites:** `12_bures_wasserstein`, `13_the_bures_bridge`, `07_mccann_geodesic`.

**What you'll be able to do**
- Build the affine McCann map between Gaussians and trace the $W_2$ geodesic (ellipses that translate, rotate, rescale).
- State the Benamou–Brenier dynamic form and Otto's Riemannian view of Wasserstein space.
- Add the rows of the Gaussian arc to the classical↔quantum dictionary and close Module 3.

One arc remains, and it closes Module 3. You have a distance on Gaussians (12) and the bridge to quantum states (13). Now meet the *geodesic* — the optimal path — in two complementary readings: McCann's explicit affine map that morphs one ellipse into another, and Benamou and Brenier's picture of mass flowing with least kinetic energy. Together they give Wasserstein space a geometry of its own, a curved one.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from qot_course import viz
from qot_course.transport.gaussian import gaussian_ot_map, gaussian_geodesic

np.random.seed(42)
viz.use_course_style()

## Where we are

Two bricks brought us here: `12_bures_wasserstein` gave the closed-form $W_2$ between Gaussians, and `13_the_bures_bridge` showed its matrix term *is* the quantum Bures distance. Now we trace the geodesic between two Gaussians — first as an explicit map, then as a flow — and add this module's last rows to the dictionary before Module 4 carries transport to density matrices.

## 1. The $W_2$ geodesic — an affine McCann map

Between Gaussians the optimal transport map is affine and explicit. For zero-mean covariances,

$$ T(x) = A\,x, \qquad A = \Sigma_0^{-1/2}\bigl(\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}\bigr)^{1/2}\Sigma_0^{-1/2}, $$

the unique symmetric positive-definite $A$ with $A\Sigma_0 A = \Sigma_1$; for general means $T(x) = m_1 + A(x - m_0)$. The $W_2$ geodesic therefore stays Gaussian,

$$ \mu_t = \mathcal{N}\big((1-t)m_0 + t\,m_1,\ M_t\,\Sigma_0\,M_t^\top\big), \qquad M_t = (1-t)I + tA, $$

rotating, translating, and rescaling the covariance ellipse all at once. Let's draw it.

In [ ]:
mean_0 = np.array([-2.5, -1.0])
cov_0  = np.array([[1.5, 0.4], [0.4, 0.6]])
mean_1 = np.array([2.5, 1.5])
cov_1  = np.array([[0.5, -0.3], [-0.3, 1.4]])

def ellipse(mean, cov, n_std=1.5, **kwargs):
    """A covariance ellipse at n_std standard deviations."""
    vals, vecs = np.linalg.eigh(0.5 * (cov + cov.T))
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = float(np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0])))
    width, height = (2.0 * n_std * np.sqrt(np.clip(vals, 0.0, None))).tolist()
    return patches.Ellipse(xy=tuple(mean), width=width, height=height, angle=angle, **kwargs)

A = gaussian_ot_map(cov_0, cov_1)
print(f"A Sigma_0 A == Sigma_1 ?  {bool(np.allclose(A @ cov_0 @ A.T, cov_1, atol=1e-9))}")
print(f"A symmetric (SPD)?        {bool(np.allclose(A, A.T) and np.all(np.linalg.eigvalsh(A) > 0))}")

ts = np.linspace(0.0, 1.0, 9)
snapshots = [gaussian_geodesic(mean_0, cov_0, mean_1, cov_1, t) for t in ts]
means = np.array([s[0] for s in snapshots])

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.colormaps["viridis"]
for k, (t, (mean_t, cov_t)) in enumerate(zip(ts, snapshots)):
    shade = k / (len(ts) - 1)
    ax.add_patch(ellipse(mean_t, cov_t,
                         edgecolor=cmap(shade), facecolor=cmap(shade),
                         alpha=0.18 + 0.5 * shade, linewidth=2.0,
                         label=f"t = {t:.2f}" if k in (0, len(ts) // 2, len(ts) - 1) else None))
ax.plot(means[:, 0], means[:, 1], "--", color=viz.COLORS["text"], lw=1.2, label="mean trajectory")
ax.scatter(means[[0, -1], 0], means[[0, -1], 1], color=viz.COLORS["text"], s=70, zorder=4)
ax.set_xlim(-6, 6)
ax.set_ylim(-5, 5)
ax.set_aspect("equal")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("W2 geodesic in the Gaussian family: translate + rotate + rescale at once", pad=12)
ax.legend(loc="upper left")
plt.show()

**Read the figure.** The map $A$ pushes $\Sigma_0$ exactly onto $\Sigma_1$ (verified above). The geodesic ellipses flow from the lower-left Gaussian (violet) to the upper-right one (yellow): the mean tracks a straight line, while the covariance shape blends so that the distribution **stays Gaussian at every $t$**. That closed-form geodesic is a luxury we have only because the family is Gaussian — and it is exactly the displacement-interpolation idea of `07_mccann_geodesic`, now in two dimensions and in matrix form.

**One more look: feed the geodesic density matrices.** In `03/13` the matrix term of $W_2$ became the Bures distance on density matrices. Does the geodesic formula $\Sigma_t = M_t\,\Sigma_0\,M_t$ become a path of quantum states in the same way? Its trace answers. Expanding $M_t = (1-t)I + tA$ gives $\mathrm{tr}\,\Sigma_t = (1-t)^2\,\mathrm{tr}\,\Sigma_0 + 2t(1-t)\,\mathrm{tr}(A\Sigma_0) + t^2\,\mathrm{tr}\,\Sigma_1$, using $A\Sigma_0A = \Sigma_1$ for the last term. By the formula for $A$, $\mathrm{tr}(A\Sigma_0) = \mathrm{tr}\sqrt{\Sigma_0^{1/2}\,\Sigma_1\,\Sigma_0^{1/2}}$, which for unit-trace inputs is the Uhlmann fidelity $F_U$ of `03/13`. Between two density matrices the trace is therefore $(1-t)^2 + 2t(1-t)\,F_U + t^2$: it equals $1$ at both ends and $(1 + F_U)/2$ at the midpoint, below $1$ whenever the two states differ. The next cell checks this on two qubit states.

In [ ]:
# Feed the Gaussian geodesic formula two density matrices (unit trace) and follow the trace.
rho_0 = np.array([[0.8, 0.1], [0.1, 0.2]])
rho_1 = np.array([[0.3, -0.2], [-0.2, 0.7]])
zero = np.zeros(2)
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    _, sigma_t = gaussian_geodesic(zero, rho_0, zero, rho_1, t)
    print(f"t = {t:.2f}   tr(Sigma_t) = {np.trace(sigma_t):.4f}")
fidelity = float(np.trace(gaussian_ot_map(rho_0, rho_1) @ rho_0))   # tr(A rho_0) = F_U
print(f"Uhlmann fidelity F_U = {fidelity:.4f},   (1 + F_U) / 2 = {(1 + fidelity) / 2:.4f}")

**Read the output.** The trace is $1$ at both ends and dips to $0.9060$ at the midpoint, which is $(1 + F_U)/2$ with $F_U = 0.8120$. Between its ends the curve leaves the set of density matrices. So the formula extends to density matrices, but the curve it draws is not a path of quantum states, and the Bures geodesic between two states, which stays among the states, is a different curve. This course names that geodesic and does not build it.

## 2. The dynamic view — Benamou–Brenier and Otto

The static problem has a fluid-dynamics twin (Benamou & Brenier, 2000):

$$ W_2^2(\mu_0, \mu_1) = \inf_{(\rho_t, v_t)}\ \int_0^1\!\!\int |v_t(x)|^2\,\rho_t(x)\,\mathrm{d}x\,\mathrm{d}t, $$

minimised over a path of distributions $\rho_t$ and a velocity field $v_t$ obeying the **continuity equation** $\partial_t \rho_t + \nabla\!\cdot(\rho_t v_t) = 0$ with $\rho_0 = \mu_0$, $\rho_1 = \mu_1$. In words: move the mass from $\mu_0$ to $\mu_1$ with the least total kinetic energy. The optimal velocity is the gradient of a Kantorovich potential, and the optimal $\rho_t$ is the McCann interpolation of `07`.

**Otto (2001)** read this as geometry: it gives the space of distributions $\mathcal{P}_2$ a **Riemannian-like metric**, with velocity fields as tangent vectors and McCann interpolations as geodesics. So Wasserstein space is *curved*. Otto computed, formally, that its curvature is non-negative. Two theorems make a statement of that kind rigorous, as non-negative curvature in Alexandrov's sense: Ambrosio, Gigli and Savaré (2005, Thm 7.3.2) when the ground space is a Euclidean space $\mathbb{R}^d$, the setting of this module, and Lott and Villani (2009) when it is a compact manifold of non-negative curvature. In that respect it resembles the Fisher–Rao geometry of `02/07`, which `02/08` turns into a piece of a sphere, but it respects the ground space rather than only the amplitudes. On density matrices, Carlen and Maas (2014) built a counterpart of this picture: a quantum continuity equation for paths of states and, on those paths, a quantum Riemannian metric that plays the part of Otto's — the bridge holds one more time. Module 4 places that construction on its map of quantum transport, in `04/03`, and does not build it.

## 3. Dictionary update — the Bures bridge, complete

The Gaussian arc adds the last rows that Module 3 brings to the classical↔quantum dictionary. Their quantum entries are of two kinds. The density matrix and the Bures distance you have already built, in `01/13` and `02/13`. The coupling map, the Bures geodesic and the Carlen–Maas metric are named for orientation and are not built in this course, as their rows say. For the geodesic row, the check at the end of §1 shows why the Gaussian formula cannot be carried over as it stands: fed two density matrices, it draws a curve whose trace dips below $1$.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| covariance matrix $\Sigma$ (PSD, finite trace) | density matrix $\rho$ (PSD, unit trace) |
| Gaussian Bures–Wasserstein $W_2$ | Bures distance on density matrices (the bridge) |
| McCann map $T(x) = m_1 + A(x - m_0)$ | quantum coupling map (not built in this course) |
| $W_2$ geodesic on Gaussians | Bures geodesic on density matrices (implicit in `02/13` / Uhlmann; not built in this course) |
| Benamou–Brenier / Otto Riemannian metric | Carlen–Maas quantum Riemannian metric (not built in this course) |

Module 3 is complete. The next module carries transport itself to density matrices.

## Your turn

1. **Eigenvectors of $A$.** Take diagonal covariances $\Sigma_0 = \mathrm{diag}(1, 4)$ and $\Sigma_1 = \mathrm{diag}(9, 1)$. Predict the eigenvalues of the OT map $A$ (hint: $\sqrt{\sigma_1/\sigma_0}$ in each axis) and check with `gaussian_ot_map`.
2. **Constant-speed flow.** Along the Gaussian geodesic, confirm $W_2(\mu_0, \mu_t)$ grows linearly in $t$ — a geodesic has constant speed, exactly as in `07`.
3. **Kinetic energy.** For the 1-D geodesic between $\mathcal{N}(0, 1)$ and $\mathcal{N}(4, 1)$ (a pure translation), argue that the Benamou–Brenier velocity field is constant and that the total kinetic energy equals $W_2^2$.

## What you built

- You built the affine McCann map between Gaussians and traced the $W_2$ geodesic — ellipses translating, rotating, and rescaling, staying Gaussian throughout.
- You met the Benamou–Brenier dynamic form and Otto's Riemannian reading of Wasserstein space.
- You added the rows of the Gaussian arc to the classical↔quantum dictionary, the last that Module 3 brings.

**Module 3 is complete.** From Monge's deterministic map, through the Kantorovich LP, the Wasserstein metric and its geodesics, the Sinkhorn algorithm and Amari's bridge, to the Gaussian closed form and the Bures bridge — you have built the whole of classical optimal transport, and the transport rows of the dictionary that will let it speak quantum. That is a substantial body of understanding, and it is the launchpad for everything ahead.

## What's next

Module 4 — **quantum optimal transport** — carries the coupling, the Kantorovich program, the Wasserstein distance and the Sinkhorn iteration to density matrices. It opens by asking *why* a genuinely quantum transport is needed at all: what the naive "diagonal" approach misses, and how the $|+\rangle$-vs-mixed lesson and the Bures bridge you built point the way to quantum Wasserstein.

## References

- J.-D. Benamou & Y. Brenier, "A computational fluid mechanics solution to the Monge–Kantorovich mass transfer problem", *Numer. Math.* 84, 375–393 (2000).
- F. Otto, "The geometry of dissipative evolution equations: the porous medium equation", *Comm. PDE* 26, 101–174 (2001). DOI:10.1081/PDE-100002243
- E. A. Carlen & J. Maas, "An analog of the 2-Wasserstein metric in non-commutative probability under which the fermionic Fokker–Planck equation is gradient flow for the entropy", *Comm. Math. Phys.* 331, 887–926 (2014). DOI:10.1007/s00220-014-2124-8
- J. Lott & C. Villani, "Ricci curvature for metric-measure spaces via optimal transport", *Ann. of Math.* 169, 903–991 (2009). DOI:10.4007/annals.2009.169.903
- L. Ambrosio, N. Gigli & G. Savaré, *Gradient Flows in Metric Spaces and in the Space of Probability Measures*, Lectures in Mathematics ETH Zürich, Birkhäuser (2005), §7.3. DOI:10.1007/b137080
- R. McCann, "A convexity principle for interacting gases", *Adv. Math.* 128, 153–179 (1997).

**Previous:** `notebooks/03_ClassicalOptimalTransport/13_the_bures_bridge.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/01_diagonal_collapse.ipynb`